# Millimetres, pixels, and whether the board is right

Every recommendation this project makes comes out of a chain of conversions:

1. a dartboard specified in **millimetres** becomes an **array of pixel scores**;
2. a player's accuracy $\sigma$ in **millimetres** becomes a covariance in **pixel units**;
3. an **aiming point in millimetres** becomes a **pixel index**;
4. the throw distribution is **sampled at pixel centres** and normalised.

If any link is wrong, the recommendations are wrong -- and they would be wrong *quietly*,
because the answers would still look plausible. This notebook checks each link, and then
checks the whole chain at once against exact geometry using Monte Carlo, which involves no
pixels at all.

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'font.size': 9, 'legend.frameon': False,
})

from darts.dartboards import DARTBOARD_CONSTANTS as C, generate_dartboard
from darts.transitions import transition_maps, gaussian_kernel
from darts.stats import gaussian_filter
from darts.utils import mm_per_pixel, region_label

## 1. The physical constants

A competition board: 451 mm across, scoring area out to 170 mm radius, treble and double
beds 8 mm wide, bull 12.7 mm across and the 25-ring 31.8 mm.

In [2]:
spec = [('inner bull radius', C['INNER_BULLSEYE_RADIUS_MM'], 6.35, '12.7 mm diameter'),
        ('25-ring radius', C['OUTER_BULLSEYE_RADIUS_MM'], 15.9, '31.8 mm diameter'),
        ('treble inner', C['TRIPLE_INNER_RADIUS'], 99.0, ''),
        ('treble outer', C['TRIPLE_OUTER_RADIUS'], 107.0, '8 mm bed'),
        ('double inner', C['DOUBLE_INNER_RADIUS'], 162.0, ''),
        ('double outer', C['DOUBLE_OUTER_RADIUS'], 170.0, '8 mm bed'),
        ('board radius', C['DARTBOARD_RADIUS_MM'], 225.5, '451 mm diameter')]
pd.DataFrame([{'constant': k, 'model (mm)': v, 'spec (mm)': w,
               'match': 'yes' if v == w else 'NO', 'note': n} for k, v, w, n in spec]
             ).set_index('constant')

,model (mm),spec (mm),match,note
constant,,,,
inner bull radius,6.35,6.35,yes,12.7 mm diameter
25-ring radius,15.90,15.90,yes,31.8 mm diameter
treble inner,99.00,99.00,yes,
treble outer,107.00,107.00,yes,8 mm bed
double inner,162.00,162.00,yes,
double outer,170.00,170.00,yes,8 mm bed
board radius,225.50,225.50,yes,451 mm diameter


All correct. One modelling simplification worth stating: **the wires have zero width**.
The radii are the edges of the scoring regions, so a dart that would have hit a wire and
deflected is instead scored as landing wherever its centre was.

## 2. The pixel grid

`mm_per_pixel(n) = 2 x 225.5 / n`, and the board array must use exactly that spacing with
the bull sitting exactly on pixel `n // 2`. If the grid spacing and `mm_per_pixel`
disagree, $\sigma$ is converted with the wrong scale; if the centre falls between pixels,
every aim point is offset by half a pixel.

In [3]:
rows = []
for px in [128, 256, 512, 1024, 2048]:
    mmpp = mm_per_pixel(px)
    board, _ = generate_dartboard(px)
    coords = (np.arange(px) - px // 2) * mmpp
    rows.append({'pixels': px, 'mm/pixel': round(mmpp, 5),
                 'coord of pixel n//2 (mm)': round(coords[px // 2], 10),
                 'score at pixel n//2': int(board[px // 2, px // 2]),
                 'treble bed (px)': round(8 / mmpp, 1),
                 'double bed (px)': round(8 / mmpp, 1)})
pd.DataFrame(rows).set_index('pixels')

,mm/pixel,coord of pixel n//2 (mm),score at pixel n//2,treble bed (px),double bed (px)
pixels,,,,,
128,3.52344,0.0,50,2.3,2.3
256,1.76172,0.0,50,4.5,4.5
512,0.88086,0.0,50,9.1,9.1
1024,0.44043,0.0,50,18.2,18.2
2048,0.22021,0.0,50,36.3,36.3


The bull sits exactly on the centre pixel at every resolution, and the scoring beds are
2.3 pixels wide at 128 and 9.1 at 512. Two pixels across an 8 mm bed is not enough to
represent it: that alone tells you 128 is too coarse.

## 3. The covariance convention -- the trap

For a spherically symmetric throw the index order of `Sigma` does not matter. As soon as
you fit a *real* player's covariance it does. The check: build the kernel from a known
`Sigma` and measure its second moments back out.

In [4]:
N = 401
o = np.arange(N) - N // 2
col = o[None, :] * np.ones((N, 1))
row = o[:, None] * np.ones((1, N))

rows = []
for name, Sig in [('diag(100, 25)', np.diag([100.0, 25.0])),
                  ('rho = +0.8', np.array([[100., 80.], [80., 100.]])),
                  ('rho = -0.8', np.array([[100., -80.], [-80., 100.]]))]:
    k = gaussian_kernel(N, Sig)
    vcc = (k * col ** 2).sum(); vrr = (k * row ** 2).sum(); vcr = (k * col * row).sum()
    rows.append({'requested Sigma': name,
                 'Var(column) = Sigma[0,0]?': f'{vcc:.2f} vs {Sig[0,0]:.0f}',
                 'Var(row) = Sigma[1,1]?': f'{vrr:.2f} vs {Sig[1,1]:.0f}',
                 'Cov = Sigma[0,1]?': f'{vcr:.2f} vs {Sig[0,1]:.0f}'})
pd.DataFrame(rows).set_index('requested Sigma')

,"Var(column) = Sigma[0,0]?","Var(row) = Sigma[1,1]?","Cov = Sigma[0,1]?"
requested Sigma,,,
"diag(100, 25)",100.00 vs 100,25.00 vs 25,-0.00 vs 0
rho = +0.8,100.00 vs 100,100.00 vs 100,80.00 vs 80
rho = -0.8,100.00 vs 100,100.00 vs 100,-80.00 vs -80


So `Sigma` is the covariance in **(x, y) = (horizontal, vertical)** order, with the usual
sign convention on the off-diagonal. A covariance fitted to throws in ordinary millimetre
coordinates can be passed straight in.

But `gaussian_filter`'s **mean** uses the opposite order:

In [5]:
blank = np.zeros((N, N))
g = gaussian_filter(blank, np.array([10.0, 0.0]), np.diag([25.0, 25.0]))
i, j = np.unravel_index(g.argmax(), g.shape)
print(f'gaussian_filter(mu=[10, 0]) peaks at (row={i}, col={j}); centre is {N//2}')
print('-> mu[0] moved the ROW (the y direction), not the column')
print()
k = gaussian_kernel(N, np.array([[100., 80.], [80., 100.]]))
g2 = gaussian_filter(blank, np.zeros(2), np.array([[100., 80.], [80., 100.]]))
print(f'the two kernel implementations agree to {np.abs(g2 / g2.sum() - k).max():.1e}')

gaussian_filter(mu=[10, 0]) peaks at (row=210, col=200); centre is 200
-> mu[0] moved the ROW (the y direction), not the column

the two kernel implementations agree to 4.3e-19


**`mu` is `(row, column)` while `Sigma` is `(x, y)`.** They are transposes of each other.
That is now documented in the `gaussian_filter` docstring; it is the one thing most likely
to silently corrupt an anisotropic analysis.

## 4. The whole chain, against exact geometry

The decisive test. Score a dart *analytically* from its millimetre coordinates -- no board
array, no pixels, no FFT -- and sample a few million throws. Then compare against what the
pixel pipeline says for the same aim point and the same $\sigma$.

In [6]:
def exact_score(x, y):
    '''Score of a dart landing at (x, y) mm from the centre. No pixels involved.'''
    r = np.hypot(x, y)
    theta = np.mod(np.arctan2(y, x) + np.pi, 2 * np.pi) - np.pi
    number = np.zeros(len(r), dtype=np.int64)
    for score, intervals in C['SEGMENTS'].items():
        m = np.zeros(len(r), dtype=bool)
        for lo, hi in intervals:
            m |= (theta < hi * np.pi) & (theta >= lo * np.pi)
        number[m] = score
    mult = np.zeros(len(r), dtype=np.int64)
    mult[((r >= C['OUTER_BULLSEYE_RADIUS_MM']) & (r < C['TRIPLE_INNER_RADIUS'])) |
         ((r >= C['TRIPLE_OUTER_RADIUS']) & (r < C['DOUBLE_INNER_RADIUS']))] = 1
    mult[(r >= C['TRIPLE_INNER_RADIUS']) & (r < C['TRIPLE_OUTER_RADIUS'])] = 3
    mult[(r >= C['DOUBLE_INNER_RADIUS']) & (r < C['DOUBLE_OUTER_RADIUS'])] = 2
    out = number * mult
    out[(r >= C['INNER_BULLSEYE_RADIUS_MM']) & (r < C['OUTER_BULLSEYE_RADIUS_MM'])] = 25
    out[r < C['INNER_BULLSEYE_RADIUS_MM']] = 50
    return out

rng = np.random.default_rng(1)
N_MC = 2_000_000
rows = []
for px in [128, 256, 512, 1024]:
    mmpp = mm_per_pixel(px)
    board, ck = generate_dartboard(px)
    # aim straight up from the centre, at whichever pixel centre is nearest 103 mm,
    # then give the Monte Carlo that exact position so aim rounding cannot confuse things
    r_px = int(round(103.0 / mmpp))
    aim = np.array([0.0, r_px * mmpp])
    for sigma_mm in [5.0, 10.0, 20.0]:
        s = rng.normal(size=(N_MC, 2)) * sigma_mm + aim
        sc = exact_score(s[:, 0], s[:, 1])
        mc, se = sc.mean(), sc.std() / np.sqrt(N_MC)
        sp = sigma_mm / mmpp
        pm, _, S = transition_maps(board, ck, sp * sp * np.eye(2))
        grid = float(pm[:, px // 2 + r_px, px // 2] @ S)
        rows.append({'pixels': px, 'sigma (mm)': sigma_mm,
                     'pixel grid': round(grid, 4), 'exact MC': round(mc, 4),
                     'MC std err': round(se, 4), 'error %': round(100 * (grid - mc) / mc, 2)})
pd.DataFrame(rows).set_index(['pixels', 'sigma (mm)'])

pixel grid  exact MC  MC std err  error %
pixels sigma (mm)                                           
128    5.0            40.5972   42.8413      0.0140    -5.24
       10.0           29.3962   29.4326      0.0138    -0.12
       20.0           17.3541   17.1957      0.0112     0.92
256    5.0            41.0180   42.8196      0.0140    -4.21
       10.0           29.0547   29.4738      0.0138    -1.42
       20.0           17.1744   17.2061      0.0112    -0.18
512    5.0            42.9006   43.0197      0.0140    -0.28
       10.0           29.4411   29.5173      0.0138    -0.26
       20.0           17.2269   17.2596      0.0112    -0.19
1024   5.0            42.8631   43.0082      0.0140    -0.34
       10.0           29.4464   29.5004      0.0138    -0.18
       20.0           17.2365   17.2470      0.0112    -0.06

This is the number that matters for trusting a recommendation:

* **128 px** and **256 px** are wrong by up to several percent, worst for accurate players
  (small $\sigma$), because a treble bed is barely two pixels wide.
* **512 px** is accurate to about 0.2--0.3% across the whole range of human accuracy.
* Going to **1024 px** buys surprisingly little.

The residual is a small *systematic* underestimate that does not shrink much with
resolution. It is boundary quantisation: each pixel is assigned a single score, so the
edges of the treble and double beds get rounded to the pixel grid, and the beds end up
slightly the wrong width. Averaged over a ring this mostly cancels, leaving a fraction of a
percent.

If that ever needs to go away, the fix is not a finer FFT but **anti-aliasing**: build each
score's mask at, say, 4x resolution and box-average it down, so a pixel straddling a bed
edge carries a fractional membership instead of being all-or-nothing. That would give
512-pixel FFT cost with something closer to 2048-pixel accuracy.

## 5. Anisotropy, end to end

Finally, confirm that a non-spherical throw spreads in the direction you would expect,
against the same exact-geometry Monte Carlo. Aiming at the bull, a horizontally wide
throw should land in the 6 (right) and 11 (left); a vertically wide one in the 20 (up)
and 3 (down).

In [7]:
px = 512
mmpp = mm_per_pixel(px)
board, ck = generate_dartboard(px)
labels = {6: '6 (right)', 11: '11 (left)', 20: '20 (up)', 3: '3 (down)'}

rows = []
for name, Sig_mm in [('wide horizontally: diag(25^2, 5^2)', np.diag([625.0, 25.0])),
                     ('wide vertically:   diag(5^2, 25^2)', np.diag([25.0, 625.0]))]:
    pm, _, S = transition_maps(board, ck, Sig_mm / mmpp ** 2)
    p = {int(s): float(v) for s, v in zip(S, pm[:, px // 2, px // 2])}
    L = np.linalg.cholesky(Sig_mm)
    sc = exact_score(*(rng.normal(size=(N_MC, 2)) @ L.T).T)
    for k in [6, 11, 20, 3]:
        rows.append({'Sigma': name, 'segment': labels[k],
                     'pixel grid': round(p.get(k, 0.0), 4),
                     'exact MC': round(float((sc == k).mean()), 4)})
pd.DataFrame(rows).pivot(index='Sigma', columns='segment',
                         values=['pixel grid', 'exact MC'])

pixel grid                             \
segment                             11 (left) 20 (up) 3 (down) 6 (right)   
Sigma                                                                      
wide horizontally: diag(25^2, 5^2)     0.1684  0.0001   0.0001    0.1684   
wide vertically:   diag(5^2, 25^2)     0.0001  0.1684   0.1684    0.0001   

                                    exact MC                             
segment                            11 (left) 20 (up) 3 (down) 6 (right)  
Sigma                                                                    
wide horizontally: diag(25^2, 5^2)    0.1686  0.0001   0.0001    0.1677  
wide vertically:   diag(5^2, 25^2)    0.0001  0.1684   0.1682    0.0001

The pixel pipeline and the exact geometry agree to the third decimal place, and the
orientation is the natural one.

## Conclusion

The millimetre-to-pixel chain is correct. The one caveat for practical use is resolution:
**work at 512 pixels or finer**. At 128 or 256 the answers are wrong by enough to change
which treble the model recommends, and it will not warn you.